In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["HF_HOME"] = "D:/roberta/hf_cache"

In [2]:
import json, re, time, pathlib, random
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
LM_STUDIO_URL  = "http://localhost:1234/v1/chat/completions"
LM_MODEL       = "sherkala"          # exact model name shown in LM Studio

TRAIN_SEED_CSV = "D:/roberta/data/kazsandra_human_train.csv"
TEST_SEED_CSV  = "D:/roberta/data/kazsandra_human_test.csv"

TRAIN_CKPT     = "D:/roberta/data/ai_train_kk_checkpoint.jsonl"
TEST_CKPT      = "D:/roberta/data/ai_test_kk_checkpoint.jsonl"

TRAIN_OUT      = "D:/roberta/data/ai_train_native_kk.csv"
TEST_OUT       = "D:/roberta/data/ai_test.csv"
HELDOUT        = "D:/roberta/data/heldout_test.csv"

SIM_THRESHOLD  = 0.85
MIN_LEN        = 10
MAX_LEN        = 300

KAZAKH_OR_RUSSIAN_RE = re.compile(r'[а-яА-ЯәіңғүұқөһӘІҢҒҮҰҚӨҺ]')

BAD_PREFIXES = [
    "жауап:", "қажетті:", "міндетті:", "бұл жерде",
    "мен сізге", "келесі шолу", "шолу:", "review:",
    "sure", "here is", "вот", "ответ:", "output:",
    "certainly", "of course", "as requested",
]

In [4]:
def _chat(messages, max_tokens=150, temperature=0.9, retries=4):
    """LM Studio request with exponential backoff on 5xx."""
    payload = {
        "model": LM_MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    for attempt in range(retries):
        try:
            r = requests.post(LM_STUDIO_URL, json=payload, timeout=60)
            if r.status_code < 500:
                r.raise_for_status()
                return r.json()["choices"][0]["message"]["content"].strip()
            time.sleep(2 ** attempt)
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("LM Studio unreachable after retries")


def _too_similar(text_a, text_b, threshold=SIM_THRESHOLD):
    """True if char-ngram cosine similarity >= threshold."""
    try:
        vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5))
        tfidf = vec.fit_transform([text_a, text_b])
        return float(cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]) >= threshold
    except Exception:
        return False


def keep_review(text, seed):
    """True if text passes all quality filters."""
    if not text or not isinstance(text, str):
        return False
    t = text.strip()
    if not (MIN_LEN <= len(t) <= MAX_LEN):
        return False
    if not KAZAKH_OR_RUSSIAN_RE.search(t):
        return False
    if any(t.lower().startswith(p.lower()) for p in BAD_PREFIXES):
        return False
    if _too_similar(t, seed):
        return False
    return True


def make_prompt(seed_text):
    """Build Sherkala chat prompt seeded from a real KazSAnDRA review."""
    return [
        {
            "role": "system",
            "content": (
                "Сен қысқа қазақ шолуларын жазатын пайдаланушысың. "
                "Нақты және бейресми жазылған 1-3 сөйлемнен тұратын шолу жаз. "
                "Тек шолу мәтінін жаз, басқа ештеңе қосып жазба."
            )
        },
        {
            "role": "user",
            "content": (
                f'Мына нақты шолуды оқы: "{seed_text}"\n'
                f'Осы тақырып туралы, осындай қысқа және бейресми стильде, '
                f'бірақ мүлде басқаша шолу жаз.'
            )
        }
    ]


def append_jsonl(path, record):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')


def load_jsonl(path):
    if not pathlib.Path(path).exists():
        return []
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

In [5]:
# Quick connectivity check — run this cell before generation
try:
    r = requests.get("http://localhost:1234/v1/models", timeout=5)
    models = r.json()
    print("LM Studio is reachable. Available models:")
    for m in models.get("data", []):
        print(" -", m["id"])
except Exception as e:
    print(f"❌ LM Studio not reachable: {e}")
    print("Load Sherkala in LM Studio and start the server, then re-run this cell.")

LM Studio is reachable. Available models:
 - sherkala
 - deepseek-coder-v2-lite-instruct
 - qwen3-vl-8b-instruct-abliterated-v2.0
 - qwen2.5-7b-instruct
 - qwen2-7b-instruct
 - meta-llama-3-8b-instruct@?
 - meta-llama-3-8b-instruct@q5_k_m
 - qwen/qwen2.5-vl-7b
 - text-embedding-nomic-embed-text-v1.5


In [6]:
train_seeds = pd.read_csv(TRAIN_SEED_CSV)['text'].dropna().tolist()
random.seed(42)
random.shuffle(train_seeds)

done = {r['seed'] for r in load_jsonl(TRAIN_CKPT)}
print(f"Resuming: {len(done)}/{len(train_seeds)} seeds already processed")

for i, seed in enumerate(train_seeds):
    if seed in done:
        continue
    try:
        output = _chat(make_prompt(seed))
        if keep_review(output, seed):
            append_jsonl(TRAIN_CKPT, {"text": output, "label": 1, "seed": seed})
            done.add(seed)
    except Exception as e:
        print(f"[{i}] Error on seed {i}: {e}")
        time.sleep(3)
    if (i + 1) % 100 == 0:
        print(f"Progress: {i+1}/{len(train_seeds)} | Accepted: {len(done)}")

print(f"\nTrain generation complete. Accepted: {len(done)}/{len(train_seeds)}")

Resuming: 0/5000 seeds already processed
Progress: 100/5000 | Accepted: 99
Progress: 200/5000 | Accepted: 196
Progress: 300/5000 | Accepted: 293
Progress: 400/5000 | Accepted: 391
Progress: 500/5000 | Accepted: 488
Progress: 600/5000 | Accepted: 584
Progress: 700/5000 | Accepted: 677
Progress: 800/5000 | Accepted: 771
Progress: 900/5000 | Accepted: 868
Progress: 1000/5000 | Accepted: 965
Progress: 1100/5000 | Accepted: 1063
Progress: 1200/5000 | Accepted: 1161
Progress: 1300/5000 | Accepted: 1260
Progress: 1400/5000 | Accepted: 1357
Progress: 1500/5000 | Accepted: 1455
Progress: 1600/5000 | Accepted: 1550
Progress: 1700/5000 | Accepted: 1644
Progress: 1800/5000 | Accepted: 1740
Progress: 1900/5000 | Accepted: 1833
Progress: 2000/5000 | Accepted: 1929
Progress: 2100/5000 | Accepted: 2026
Progress: 2200/5000 | Accepted: 2120
Progress: 2300/5000 | Accepted: 2216
Progress: 2400/5000 | Accepted: 2310
Progress: 2500/5000 | Accepted: 2409
Progress: 2600/5000 | Accepted: 2505
Progress: 2700/50

In [7]:
train_records = load_jsonl(TRAIN_CKPT)
train_df = pd.DataFrame(train_records)[['text', 'label']]

print(f"Rows: {len(train_df)}")
print(f"All label=1: {(train_df['label'] == 1).all()}")
lengths = train_df['text'].str.len()
print(f"Length — mean: {lengths.mean():.1f} | median: {lengths.median():.1f} | min: {lengths.min()} | max: {lengths.max()}")
print(f"Short (≤60): {(lengths <= 60).sum()} ({100*(lengths<=60).mean():.1f}%)")

assert len(train_df) > 3000, f"Too few samples ({len(train_df)}). Check LM Studio or filter thresholds."
assert (train_df['label'] == 1).all(), "Non-AI labels found in train set."
assert lengths.mean() < 250, f"Mean length {lengths.mean():.1f} unexpectedly high — check MAX_LEN."
print("\n✅ Train checkpoint verification passed")

Rows: 4832
All label=1: True
Length — mean: 111.0 | median: 101.0 | min: 10 | max: 300
Short (≤60): 1122 (23.2%)

✅ Train checkpoint verification passed


In [8]:
test_seeds = pd.read_csv(TEST_SEED_CSV)['text'].dropna().tolist()
random.seed(99)
random.shuffle(test_seeds)

done_test = {r['seed'] for r in load_jsonl(TEST_CKPT)}
print(f"Resuming: {len(done_test)}/{len(test_seeds)} seeds already processed")

for i, seed in enumerate(test_seeds):
    if seed in done_test:
        continue
    try:
        output = _chat(make_prompt(seed))
        if keep_review(output, seed):
            append_jsonl(TEST_CKPT, {"text": output, "label": 1, "seed": seed})
            done_test.add(seed)
    except Exception as e:
        print(f"[{i}] Error: {e}")
        time.sleep(3)
    if (i + 1) % 100 == 0:
        print(f"Progress: {i+1}/{len(test_seeds)} | Accepted: {len(done_test)}")

print(f"\nTest generation complete. Accepted: {len(done_test)}/{len(test_seeds)}")

Resuming: 0/2000 seeds already processed
Progress: 100/2000 | Accepted: 97
Progress: 200/2000 | Accepted: 194
Progress: 300/2000 | Accepted: 291
Progress: 400/2000 | Accepted: 390
Progress: 500/2000 | Accepted: 489
Progress: 600/2000 | Accepted: 586
Progress: 700/2000 | Accepted: 683
Progress: 800/2000 | Accepted: 777
Progress: 900/2000 | Accepted: 870
Progress: 1000/2000 | Accepted: 963
Progress: 1100/2000 | Accepted: 1060
Progress: 1200/2000 | Accepted: 1153
Progress: 1300/2000 | Accepted: 1248
Progress: 1400/2000 | Accepted: 1346
Progress: 1500/2000 | Accepted: 1442
Progress: 1600/2000 | Accepted: 1540
Progress: 1700/2000 | Accepted: 1637
Progress: 1800/2000 | Accepted: 1733
Progress: 1900/2000 | Accepted: 1826
Progress: 2000/2000 | Accepted: 1921

Test generation complete. Accepted: 1921/2000


In [9]:
test_records = load_jsonl(TEST_CKPT)
test_df = pd.DataFrame(test_records)[['text', 'label']]

print(f"Rows: {len(test_df)}")
lengths = test_df['text'].str.len()
print(f"Length — mean: {lengths.mean():.1f} | median: {lengths.median():.1f}")

assert len(test_df) > 1000, f"Too few test samples ({len(test_df)}). Check LM Studio."
assert (test_df['label'] == 1).all(), "Non-AI labels found in test set."
print("✅ Test checkpoint verification passed")

# Verify zero overlap between train and test seed pools
train_seeds_set = set(pd.read_csv(TRAIN_SEED_CSV)['text'].dropna().tolist())
test_seeds_set  = set(pd.read_csv(TEST_SEED_CSV)['text'].dropna().tolist())
overlap = train_seeds_set & test_seeds_set
assert len(overlap) == 0, f"Seed pool overlap found: {len(overlap)} shared rows — leakage risk!"
print(f"✅ No overlap between train and test seed pools ({len(train_seeds_set)} vs {len(test_seeds_set)} seeds)")

Rows: 1921
Length — mean: 112.0 | median: 101.0
✅ Test checkpoint verification passed
✅ No overlap between train and test seed pools (4999 vs 1999 seeds)


In [10]:
train_df.to_csv(TRAIN_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)
print(f"Saved {TRAIN_OUT}: {len(train_df)} rows")
print(f"Saved {TEST_OUT}:  {len(test_df)} rows")

human_test = pd.read_csv(TEST_SEED_CSV)[['text']].copy()
human_test['label'] = 0

ai_test = test_df[['text', 'label']].copy()

# Balanced 50/50
n = min(len(human_test), len(ai_test))
heldout = pd.concat(
    [human_test.sample(n, random_state=42), ai_test.sample(n, random_state=42)],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

heldout.to_csv(HELDOUT, index=False)

print(f"heldout_test.csv saved: {len(heldout)} rows")
print(heldout['label'].value_counts().to_string())
assert heldout['label'].value_counts()[0] == heldout['label'].value_counts()[1], "Heldout not balanced!"
print("✅ Heldout rebuild passed — balanced 50/50")

Saved D:/roberta/data/ai_train_native_kk.csv: 4832 rows
Saved D:/roberta/data/ai_test.csv:  1921 rows
heldout_test.csv saved: 3842 rows
label
1    1921
0    1921
✅ Heldout rebuild passed — balanced 50/50


In [11]:
human_mean = pd.read_csv(TEST_SEED_CSV)['text'].str.len().mean()
ai_mean    = test_df['text'].str.len().mean()
ratio      = ai_mean / human_mean

print("=== Final Health Stats ===")
for name, df in [("AI train", train_df), ("AI test", test_df), ("Heldout", heldout)]:
    L = df['text'].str.len()
    print(f"{name:10s}: n={len(df):5d} | mean={L.mean():.1f} | median={L.median():.1f} | min={L.min()} | max={L.max()}")

print(f"\nHealth gate — AI/human length ratio: {ai_mean:.1f}/{human_mean:.1f} = {ratio:.2f}")
assert 0.7 <= ratio <= 3.0, f"Health gate FAIL: ratio={ratio:.2f} out of range 0.7–3.0"
if ratio > 2.0:
    print(f"⚠️  Ratio {ratio:.2f} above 2.0 — review generated text lengths")
else:
    print(f"✅ Health gate passed (ratio={ratio:.2f}, target 0.7–2.0)")

=== Final Health Stats ===
AI train  : n= 4832 | mean=111.0 | median=101.0 | min=10 | max=300
AI test   : n= 1921 | mean=112.0 | median=101.0 | min=10 | max=300
Heldout   : n= 3842 | mean=86.8 | median=67.5 | min=3 | max=810

Health gate — AI/human length ratio: 112.0/61.4 = 1.82
✅ Health gate passed (ratio=1.82, target 0.7–2.0)
